**Pregunta central: si un inversor particular hubiera invertido hace 10 años, ¿qué estrategia le habría ido mejor (todo de golpe, aportaciones mensuales o concentrarse en un sector) y cuánto riesgo habría soportado?**

Preguntas de negocio:
### Preguntas que quiero responder

1. ¿Quién sube más: el índice o la empresa típica del S&P 500?
2. ¿Cuántas empresas superan al índice, y en qué sectores?
3. ¿Qué sectores han dado más rentabilidad y cuáles han asumido más riesgo?
4. ¿Es mejor invertir todo de golpe o repartirlo en aportaciones mensuales?
5. ¿Cuánto tiempo hay que mantener la inversión para no perder dinero?


In [ ]:
# Instalamos (o actualizamos a la última versión) la librería que descarga datos de bolsa
%pip install -U yfinance

In [1]:
# Importamos yfinance y le damos el nombre corto "yf" para escribir menos
import yfinance as yf

# Descargamos los precios de Apple (AAPL) desde el 1 de enero de 2024 hasta hoy
# auto_adjust=True -> ajusta los precios por dividendos y divisiones de acciones
# progress=False  -> oculta la barra de progreso para que la salida quede limpia
prueba = yf.download("AAPL", start="2024-01-01", auto_adjust=True, progress=False)

# Mostramos las 5 primeras filas para comprobar que ha funcionado
prueba.head()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2024-01-02,183.403992,186.170269,181.675070,184.895799,82488700
2024-01-03,182.030762,183.641134,181.220631,182.001124,58414500
2024-01-04,179.718948,180.884728,178.701356,179.956048,71983600
2024-01-05,178.997696,180.558667,177.999867,179.797952,62379700
2024-01-08,183.324966,183.364493,179.313871,179.896761,59144500


In [2]:
# Importamos las librerías necesarias
import pandas as pd              # para trabajar con tablas
import requests                  # para descargar páginas web
import os                        # para crear carpetas si no existen
from io import StringIO          # para que pandas lea el texto de la web como si fuera un archivo

# Compruebo dónde se está ejecutando el notebook
try:
    # Si estoy en Colab, conecto mi Drive y uso mi carpeta
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_PROYECTO = "/content/drive/MyDrive/PORTFOLIO MARCOS GUERRERO SANCHEZ/analisis-estrategias-inversion/"
except ImportError:
    # Si no estoy en Colab, uso la carpeta donde esté el proyecto descargado
    # El ../ sube un nivel, porque el notebook está dentro de Notebooks/
    RUTA_PROYECTO = "../"

# Carpeta donde guardaremos los datos originales
RUTA = RUTA_PROYECTO + "Data/Raw/"

# Creo la carpeta si no existe, para que funcione en cualquier ordenador
os.makedirs(RUTA, exist_ok=True)

# Fecha desde la que queremos descargar los precios
FECHA_INICIO = "2016-01-01"

# Fecha de hoy. yfinance no incluye el día que pones como fin,
# así que los datos llegan hasta ayer y nunca descargo un día que no ha cerrado
FECHA_FIN = pd.Timestamp.today().strftime("%Y-%m-%d")

# Dirección de la página de Wikipedia con la lista de empresas del S&P 500
URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Descargamos la página. El "User-Agent" hace que Wikipedia nos trate como un navegador normal
# (sin él, a veces bloquea la petición)
respuesta = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})

# pandas busca todas las tablas de la página; la primera [0] es la de las empresas
tabla_wiki = pd.read_html(StringIO(respuesta.text))[0]

# Nos quedamos solo con las columnas que nos interesan
empresas = tabla_wiki[["Symbol", "Security", "GICS Sector", "GICS Sub-Industry"]].copy()

# Les ponemos nombres en español y más cómodos de usar
empresas.columns = ["ticker", "nombre", "sector", "industria"]

# Yahoo Finance usa guion en vez de punto en algunos tickers (BRK.B -> BRK-B)
empresas["ticker"] = empresas["ticker"].str.replace(".", "-", regex=False)

# Comprobamos cuántas empresas hay y vemos las primeras filas
print("Los datos se guardarán en:", RUTA)
print("Número de empresas:", len(empresas))
empresas

Mounted at /content/drive
Los datos se guardarán en: /content/drive/MyDrive/PORTFOLIO MARCOS GUERRERO SANCHEZ/analisis-estrategias-inversion/Data/Raw/
Número de empresas: 503


,ticker,nombre,sector,industria
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services
...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment


In [ ]:
# Comprobar cuantas empresas hay en cada sector
empresas["sector"].value_counts()

,count
sector,
Industrials,83
Financials,76
Information Technology,75
Health Care,60
Consumer Discretionary,47
Consumer Staples,34
Utilities,31
Real Estate,30
Materials,24


In [ ]:
# Importamos las librerías necesarias
import yfinance as yf            # para descargar datos de bolsa
import time                      # para hacer pausas entre descargas

# Lista vacía donde iremos guardando la tabla de precios de cada empresa
lista_precios = []

# Lista donde apuntaremos las empresas que den error, para revisarlas después
fallidos = []

# Recorremos las empresas una a una; enumerate nos da también su posición (0, 1, 2...)
for posicion, ticker in enumerate(empresas["ticker"]):

    # Intentamos descargar; si algo falla, saltamos al "except" en vez de parar todo
    try:
        # Descargamos los precios diarios de esta empresa desde FECHA_INICIO hasta FECHA_FIN
        # FECHA_FIN es hoy, y yfinance no incluye ese día, así que solo descargo días ya cerrados
        df = yf.download(ticker, start=FECHA_INICIO, end=FECHA_FIN, auto_adjust=True,
                         progress=False, multi_level_index=False)

        # Si la descarga vino vacía, la apuntamos como fallida y pasamos a la siguiente
        if df.empty:
            fallidos.append(ticker)
            continue

        # Añadimos una columna con el ticker para saber de qué empresa es cada fila
        df["ticker"] = ticker

        # Guardamos esta tabla en la lista
        lista_precios.append(df)

    except Exception as error:
        # Si hubo un error, apuntamos la empresa y seguimos con la siguiente
        fallidos.append(ticker)

    # Pausa de 0,2 segundos para no saturar a Yahoo Finance con peticiones
    time.sleep(0.2)

    # Cada 50 empresas mostramos por dónde vamos
    if (posicion + 1) % 50 == 0:
        print(f"Descargadas {posicion + 1} de {len(empresas)}")

# Juntamos todas las tablas en una sola y convertimos la fecha en columna normal
precios = pd.concat(lista_precios).reset_index()

# Descargamos SPY, el fondo que replica el S&P 500, para usarlo de referencia
# También hasta FECHA_FIN, para que el índice termine el mismo día que las empresas
indice = yf.download("SPY", start=FECHA_INICIO, end=FECHA_FIN, auto_adjust=True,
                     progress=False, multi_level_index=False).reset_index()

# Resumen de la descarga
print("Empresas descargadas:", len(lista_precios))
print("Empresas con error:", len(fallidos), fallidos)

Descargadas 50 de 503
Descargadas 100 de 503
Descargadas 150 de 503
Descargadas 200 de 503
Descargadas 250 de 503
Descargadas 300 de 503
Descargadas 350 de 503
Descargadas 400 de 503
Descargadas 450 de 503
Descargadas 500 de 503
Empresas descargadas: 503
Empresas con error: 0 []


In [ ]:
# Mostramos el tamaño de la tabla: (número de filas, número de columnas)
print("Tamaño de la tabla:", precios.shape)

# Mostramos la fecha más antigua y la más reciente de todos los datos
print("Desde:", precios["Date"].min(), "hasta:", precios["Date"].max())

# Contamos cuántas filas (días) tiene cada empresa
filas_por_empresa = precios.groupby("ticker").size()

# Mostramos la empresa con menos días y la que tiene más
print("Mínimo de días por empresa:", filas_por_empresa.min())
print("Máximo de días por empresa:", filas_por_empresa.max())

# Mostramos las 5 primeras filas para ver cómo son los datos
precios.head()

Tamaño de la tabla: (1315666, 7)
Desde: 2016-01-04 00:00:00 hasta: 2026-10-09 00:00:00
Mínimo de días por empresa: 7
Máximo de días por empresa: 2708


,Date,Close,High,Low,Open,Volume,ticker
0,2016-01-04,87.046112,87.935427,86.204231,87.775355,3919531,MMM
1,2016-01-05,87.425552,87.449269,86.328732,87.046111,3214968,MMM
2,2016-01-06,85.664711,86.417665,85.030335,86.316874,3584532,MMM
3,2016-01-07,83.577797,84.858412,83.376220,84.496758,4249986,MMM
4,2016-01-08,83.293213,84.484895,83.133141,83.809015,3186144,MMM


In [ ]:
# 10 empresas con menos días de datos.
print("10 empresas con menos días de datos")
filas_por_empresa.sort_values().head(10)

10 empresas con menos días de datos


,0
ticker,
VYLR,7
HONA,82
FDXF,95
Q,240
SNDK,416
GEV,637
SOLV,638
RDDT,641
VLTO,757


In [ ]:
# Guardamos las tres tablas como CSV en la carpeta Data/Raw
# to_csv() convierte la tabla en un archivo CSV
# index=False evita que se añada una columna extra con la numeración de las filas
empresas.to_csv(RUTA + "empresas.csv", index=False)
precios.to_csv(RUTA + "precios.csv", index=False)
indice.to_csv(RUTA + "indice.csv", index=False)

# Confirmamos el guardado mostrando el tamaño de cada tabla
print("Empresas:", empresas.shape)
print("Precios:", precios.shape)
print("Índice:", indice.shape)

Empresas: (503, 4)
Precios: (1315666, 7)
Índice: (2708, 6)


In [3]:
# Descargo el histórico completo del índice, desde que existe el ETF SPY
# Lo uso solo en el notebook 05, para poder analizar plazos de 10 y 20 años
indice_largo = yf.download("SPY", start="1993-01-01", end=FECHA_FIN,
                           auto_adjust=True, progress=False, multi_level_index=False)

# Convierto la fecha de índice a columna, igual que en el resto de descargas
indice_largo = indice_largo.reset_index()

# Lo guardo en Data/Raw
indice_largo.to_csv(RUTA + "indice_largo.csv", index=False)

print("Histórico largo:", indice_largo.shape)
print("Desde:", indice_largo["Date"].min().date(), "hasta:", indice_largo["Date"].max().date())

Histórico largo: (8482, 6)
Desde: 1993-01-29 hasta: 2026-10-09
